## Importaciones de Librerías

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


import optuna
import xgboost as xgb

import joblib

## Carga de Datos

In [ ]:
df = pd.read_csv('../data/clean/prueba.csv')

## Preparación variables X, y

In [ ]:
feature_var = [name for name in df.columns if name !='selected']
X = df[feature_var]
y = df['selected']

## Separación datos en entreno y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

## Búsuqueda de Mejores Hiperparametros con Optuna

In [ ]:
def objective_xgb(trial):
    params = {
        'max_depth' : trial.suggest_int('max_depth', 5, 20),
        'eta': trial.suggest_float('eta', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'subsample': trial.suggest_float('subsample', 0.3, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'eval_metric': 'logloss',
        'random_state': 42
    }

    model = xgb.XGBClassifier(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=50)
print("XGBoost - Mejores hiperparámetros:", study_xgb.best_params)

## Pipeline

In [ ]:
pipeline = Pipeline(
    steps=[
        ('model', xgb.XGBClassifier(**study_xgb.best_params, eval_metric='logloss', random_state=42))
    ]
)

In [ ]:
print("Entrenando el modelo final con los mejores parámetros...")
pipeline.fit(X_train, y_train)

In [ ]:
# Calcular predicciones para ambos conjuntos
y_train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calcular el ROC-AUC para ambos
auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print(f"ROC-AUC Entrenamiento: {auc_train:.4f}")
print(f"ROC-AUC Prueba (Test):  {auc_test:.4f}")

# Calcular la brecha (gap)
brecha = auc_train - auc_test
print(f"Brecha de Overfitting:   {brecha:.4f}")

In [ ]:
print("\nEvaluando el modelo en el conjunto de prueba (X_test)...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score en Test: {roc_auc_score(y_test, y_pred_proba):.4f}")

## Guardado del modelo

In [ ]:
joblib.dump(pipeline, '../models/xgb_model.pkl')